# Microsoft Foundry Hosted Agents (Custom Frameworks)

This notebook deploys the shared lab infrastructure using `main.bicep`:

- Azure API Management (APIM)
- Microsoft Foundry instances (One foundry instance hosting models + One found instance hosting agents)
- GPT-5-Mini model deployment

## Why run custom frameworks on Foundry Hosted Agents?

1. Built-in observability, tracing, and monitoring.
Foundry provides a standard operational surface for agent runs, telemetry, and diagnostics so teams can troubleshoot faster and keep a consistent monitoring model across different agent runtimes.

2. Agent Identity and RBAC by default.
Your runtime is registered as a Foundry Agent and gets an Agent Identity, enabling least-privilege access to downstream Azure resources (for example, storage, search, or APIs) through RBAC instead of embedded secrets.

3. Foundry guardrails and governance.
Hosted agents can inherit platform safety controls and governance policies, helping you enforce security and compliance consistently even when the runtime framework is custom.

4. Discovery through Agent365.
Publishing into the Foundry ecosystem makes agents easier to discover and reuse across teams, reducing duplicate implementations.

5. Native evaluation and risk testing integration.
You can plug directly into Foundry evaluations, red teaming, and cost-estimation workflows to compare quality, safety, and spend using the same platform tooling.

6. Control plane and platform operations.
Agents are managed as platform assets in the Foundry control plane, with operational benefits such as managed hosting lifecycle, scaling, and centralized administration.

## When custom frameworks are a good fit

- You need framework-specific capabilities (for example, Pydantic AI or Strands primitives) not available in a default runtime.
- You want framework flexibility without giving up Foundry governance and enterprise operations.
- You need to standardize deployment and operations across multiple agent implementations.

After deployment, continue with one framework-specific setup notebook:

- `src/responses/agents/pydantic-ai/01_hosted_agent_pydantic_setup.ipynb`
- `src/responses/agents/strands/01_hosted_agent_strands_setup.ipynb`

In [6]:
import json
import os
import subprocess
from pathlib import Path

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f'Command failed: {result.returncode}')
    return result

lab_dir = Path.cwd()
if not (lab_dir / 'main.bicep').exists():
    raise FileNotFoundError('Run this notebook from labs/ai-foundry-hosted-agents-custom-framework')

print(f'Lab directory: {lab_dir}')

Lab directory: c:\GitHub\ai-gateway\AI-Gateway\labs\ai-foundry-hosted-agents-custom-framework


In [1]:
# Update these values as needed
subscription_id = '9be21359-7b5a-4ccd-b0c7-24c97acd7d0c'
resource_group = 'rg-aigw-hosted-custom-framework'
location = 'swedencentral'
deployment_name = 'deploy-hosted-custom-framework'

# Two Foundry resources:
# - index 0: model hosting
# - index 1: hosted-agent hosting
ai_services_config = [
    {
        'name': 'foundry-models',
        'location': location,
        'priority': 1,
        'weight': 100
    },
    {
        'name': 'foundry-agents',
        'location': location,
        'priority': 1,
        'weight': 100
    }
]

# gpt-5-mini is available in this subscription/region and was deployed manually.
# Keep the version aligned with the manually deployed model.
models_config = [
    {
        'aiservice': 'foundry-models',
        'name': 'gpt-5-mini',
        'publisher': 'OpenAI',
        'version': '2025-08-07',
        'sku': 'GlobalStandard',
        'capacity': 10
    }
]

apim_subscriptions = [
    {
        'name': 'starter',
        'displayName': 'Starter',
        'scope': '/apis'
    }
]

# Deploy the hosted agent first. After it is running, set these values and rerun
# this notebook to add the APIM proxy in front of the Hosted Agent Responses API.
enable_hosted_agent_responses_api = True
hosted_agent_id = 'strands-agent:3'  # set to your deployed hosted agent id to enable APIM proxy

print('Configuration ready. Set real values before running deployment.')

Configuration ready. Set real values before running deployment.


In [7]:
from pathlib import Path
import json
import shutil

# Run Azure CLI lookups for Foundry user object IDs and save the results locally.
# Update the user_principal_names list with any users you want to grant Foundry User access.
user_principal_names = [
    # 'user@contoso.com',
]

az_cmd = shutil.which('az.cmd') or shutil.which('az')
if not az_cmd:
    raise FileNotFoundError('Azure CLI was not found on PATH. Install Azure CLI or add az.cmd to PATH.')


def az_cli(args):
    command = ['cmd', '/c', az_cmd, *args] if az_cmd.lower().endswith('.cmd') else [az_cmd, *args]
    return run(command).stdout.strip()

signed_in_user_id = az_cli(['ad', 'signed-in-user', 'show', '--query', 'id', '-o', 'tsv'])
user_object_ids = []
if signed_in_user_id:
    user_object_ids.append(signed_in_user_id)

for user_principal_name in user_principal_names:
    user_id = az_cli(['ad', 'user', 'show', '--id', user_principal_name, '--query', 'id', '-o', 'tsv'])
    if user_id:
        user_object_ids.append(user_id)

foundry_user_object_ids = list(dict.fromkeys(user_object_ids))

output_file = lab_dir / 'foundry-user-object-ids.json'
output_file.write_text(json.dumps(foundry_user_object_ids, indent=2), encoding='utf-8')

print('Foundry user object IDs:')
print(json.dumps(foundry_user_object_ids, indent=2))
print(f'Saved to: {output_file}')


$ cmd /c C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd ad signed-in-user show --query id -o tsv
13f94c0d-8a51-492a-928b-59392c23c1ac

Foundry user object IDs:
[
  "13f94c0d-8a51-492a-928b-59392c23c1ac"
]
Saved to: c:\GitHub\ai-gateway\AI-Gateway\labs\ai-foundry-hosted-agents-custom-framework\foundry-user-object-ids.json


In [9]:
# Create resource group and deploy Bicep
# Ensure Azure CLI directory is available to subprocess calls in this kernel

import os

az_dir = str(Path(az_cmd).parent)
if az_dir not in os.environ.get('PATH', ''):
    os.environ['PATH'] = az_dir + os.pathsep + os.environ.get('PATH', '')

def az_run(args):
    command = ['cmd', '/c', az_cmd, *args] if az_cmd.lower().endswith('.cmd') else [az_cmd, *args]
    return run(command)

az_run(['account', 'set', '--subscription', subscription_id])
az_run(['group', 'create', '--name', resource_group, '--location', location])

parameters_file = lab_dir / 'main.parameters.generated.json'
parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'aiServicesConfig': {'value': ai_services_config},
        'modelsConfig': {'value': models_config},
        'apimSku': {'value': 'Basicv2'},
        'apimSubscriptionsConfig': {'value': apim_subscriptions},
        'inferenceAPIPath': {'value': 'inference'},
        'inferenceAPIType': {'value': 'AzureAI'},
        'foundryProjectName': {'value': 'default'},
        'foundryAgentAiServiceIndex': {'value': 1},
        'foundryUserObjectIds': {'value': foundry_user_object_ids},
        'enableHostedAgentResponsesApi': {'value': enable_hosted_agent_responses_api},
        'hostedAgentId': {'value': hosted_agent_id},
        'hostedAgentResponsesApiPath': {'value': 'hosted-agent-responses'},
        'hostedAgentResponsesApiVersion': {'value': '2025-05-15-preview'}
    }
}

parameters_file.write_text(json.dumps(parameters, indent=2), encoding='utf-8')
print(f'Wrote parameters: {parameters_file}')

# Ensure Bicep is available, then validate before create
az_run(['bicep', 'install'])

template_file = str((lab_dir / 'main.bicep').resolve())
parameters_arg = f"@{parameters_file.resolve()}"

# Validate first, but do not stop the notebook on transient/CLI validation failures.
try:
    az_run([
        'deployment', 'group', 'validate',
        '--resource-group', resource_group,
        '--template-file', template_file,
        '--parameters', parameters_arg,
        '--only-show-errors'
    ])
except RuntimeError as ex:
    print(f'Validation failed, continuing to deployment create step: {ex}')

result = az_run([
    'deployment', 'group', 'create',
    '--resource-group', resource_group,
    '--name', deployment_name,
    '--template-file', template_file,
    '--parameters', parameters_arg,
    '--only-show-errors'
])


$ cmd /c C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd account set --subscription 9be21359-7b5a-4ccd-b0c7-24c97acd7d0c

$ cmd /c C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd group create --name rg-aigw-hosted-custom-framework --location swedencentral
{
  "id": "/subscriptions/9be21359-7b5a-4ccd-b0c7-24c97acd7d0c/resourceGroups/rg-aigw-hosted-custom-framework",
  "location": "swedencentral",
  "managedBy": null,
  "name": "rg-aigw-hosted-custom-framework",
  "properties": {
    "provisioningState": "Succeeded"
  },
  "tags": null,
  "type": "Microsoft.Resources/resourceGroups"
}

Wrote parameters: c:\GitHub\ai-gateway\AI-Gateway\labs\ai-foundry-hosted-agents-custom-framework\main.parameters.generated.json

$ cmd /c C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.cmd bicep install
Bicep CLI is already installed at 'C:\Users\ollisgeorge\.azure\bin\bicep.exe'. Skipping installation as no specific version was requested.


$ cmd /c C:\Program Files\Microsoft SDKs\Azure

RuntimeError: Command failed: 1

In [ ]:
# Print key outputs
outputs = run([
    'az', 'deployment', 'group', 'show',
    '--resource-group', resource_group,
    '--name', deployment_name,
    '--query', 'properties.outputs',
    '-o', 'json'
]).stdout
print(json.dumps(json.loads(outputs), indent=2))

## Next step

1. Run this notebook once to deploy Foundry and APIM.
2. Use one of the framework notebooks under `src/responses/agents/` to build, push, and deploy the hosted agent.
3. Set `hosted_agent_id` and `enable_hosted_agent_responses_api=True` in the config cell.
4. Re-run this notebook to add the APIM front door for the Hosted Agent Responses API.
5. Call APIM at the `hostedAgentResponsesApimPath` output endpoint.